# Split Dataset

Phase 5 — chronological train/test split of `data/featured/hourly`.
See [01-pyspark-warehouse.md](../../docs/de/01-pyspark-warehouse.md).

In [ ]:
# session + paths
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder.appName("split")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "spark-jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    # pinned so the executors have a fixed address to call back on
    .config("spark.driver.port", "7078")
    .config("spark.blockManager.port", "7079")
    .config("spark.cores.max", 2)
    .config("spark.executor.memory", "1g")
    .config("spark.sql.shuffle.partitions", 64)
    .getOrCreate()
)

print("app id :", spark.sparkContext.applicationId)

FEATURED = "/opt/data/featured"

# Chronological — a random split would leak future demand into training.
# 2019-2022 train / 2023 test is roughly 80/20 by row count and keeps the test
# year clear of the COVID years, which behave unlike any other in the series.
TRAIN_YEARS = [2019, 2020, 2021, 2022]
TEST_YEARS = [2023]

hourly = spark.read.parquet(f"{FEATURED}/hourly")
print(f"hourly : {hourly.count():,} rows")
hourly.groupBy("year").count().orderBy("year").show()


## Drop the incomplete-history head

`lag_168h` needs seven days of prior hours, so the first week of each station's
history has no usable lag features. Those rows are dropped from the head of the
train split rather than imputed — filling them would invent demand history.

In [2]:
# drop rows whose lag features fall outside the available history
LAG_COLUMNS = ["lag_1h", "lag_24h", "lag_168h", "roll_mean_24h", "roll_mean_168h"]

before = hourly.count()
usable = hourly.dropna(subset=LAG_COLUMNS)
after = usable.count()

print(f"rows before : {before:,}")
print(f"dropped     : {before - after:,}  (incomplete lag history)")
print(f"rows after  : {after:,}")

# the drop should land entirely in the first week of the earliest year
usable.agg(
    F.min("start_date").alias("first_usable_date"),
    F.max("start_date").alias("last_date"),
).show()

rows before : 64,465,104
dropped     : 143,808  (incomplete lag history)
rows after  : 64,321,296
+-----------------+----------+
|first_usable_date| last_date|
+-----------------+----------+
|       2019-01-04|2023-12-31|
+-----------------+----------+



## Split and write

In [3]:
# chronological split on the calendar year
available = {r.year for r in usable.select("year").distinct().collect()}
missing = (set(TRAIN_YEARS) | set(TEST_YEARS)) - available
assert not missing, (
    f"years {sorted(missing)} are not in {FEATURED}/hourly (found {sorted(available)}) — "
    "re-run etl.ipynb then ml-ds.ipynb before splitting"
)

train = usable.where(F.col("year").isin(TRAIN_YEARS))
test = usable.where(F.col("year").isin(TEST_YEARS))

train.write.mode("overwrite").partitionBy("year", "month").parquet(f"{FEATURED}/train")
test.write.mode("overwrite").partitionBy("year", "month").parquet(f"{FEATURED}/test")

train = spark.read.parquet(f"{FEATURED}/train")
test = spark.read.parquet(f"{FEATURED}/test")

train_n, test_n = train.count(), test.count()
total = train_n + test_n
print(f"train : {train_n:>10,} rows  ({train_n / total:.1%})  -> {FEATURED}/train")
print(f"test  : {test_n:>10,} rows  ({test_n / total:.1%})  -> {FEATURED}/test")


train : 51,435,336 rows  (80.0%)  -> /opt/data/featured/train
test  : 12,885,960 rows  (20.0%)  -> /opt/data/featured/test


## Validate and record

In [4]:
# validation
checks = []

checks.append(("splits partition the data", train_n + test_n == after))
checks.append(("train is non-empty", train_n > 0))
checks.append(("test is non-empty", test_n > 0))
checks.append(("train share is 70-85%", 0.70 <= train_n / (train_n + test_n) <= 0.85))

# no temporal overlap — every train date must precede every test date
train_max = train.agg(F.max("start_date")).first()[0]
test_min = test.agg(F.min("start_date")).first()[0]
print(f"train ends   : {train_max}")
print(f"test starts  : {test_min}")
checks.append(("no temporal overlap", train_max < test_min))

# lag features must be complete in both splits
for name, df in (("train", train), ("test", test)):
    nulls = df.where(
        F.col("lag_168h").isNull() | F.col("roll_mean_168h").isNull()
    ).count()
    checks.append((f"{name} lags complete", nulls == 0))

print()
for name, passed in checks:
    print(f"  [{'PASS' if passed else 'FAIL'}] {name}")

# target mean/variance per split, for reference when reading model scores
print("\ntarget summary")
summary = None
for name, df in (("train", train), ("test", test)):
    row = df.agg(
        F.lit(name).alias("split"),
        F.count("*").alias("rows"),
        F.mean("trip_count").alias("mean"),
        F.variance("trip_count").alias("variance"),
        F.max("trip_count").alias("max"),
        F.mean((F.col("trip_count") == 0).cast("double")).alias("zero_share"),
    )
    summary = row if summary is None else summary.unionByName(row)

summary.show(truncate=False)

# yearly volume — the COVID years should stand out against 2019 and 2022/2023
print("trips per year")
(
    train.withColumn("split", F.lit("train"))
    .unionByName(test.withColumn("split", F.lit("test")))
    .groupBy("year", "split")
    .agg(
        F.sum("trip_count").alias("trips"),
        F.mean("trip_count").alias("mean_per_station_hour"),
    )
    .orderBy("year")
    .show()
)


train ends   : 2022-12-31
test starts  : 2023-01-01

  [PASS] splits partition the data
  [PASS] train is non-empty
  [PASS] test is non-empty
  [PASS] train share is 70-85%
  [PASS] no temporal overlap
  [PASS] train lags complete
  [PASS] test lags complete

target summary
+-----+--------+------------------+------------------+---+------------------+
|split|rows    |mean              |variance          |max|zero_share        |
+-----+--------+------------------+------------------+---+------------------+
|train|51435336|0.715259875039992 |5.261658593511779 |170|0.7872846791552018|
|test |12885960|0.8342587591456128|3.8352679556846048|112|0.6922098935585708|
+-----+--------+------------------+------------------+---+------------------+

trips per year
+----+-----+--------+---------------------+
|year|split|   trips|mean_per_station_hour|
+----+-----+--------+---------------------+
|2019|train| 9682128|   0.7598502984425237|
|2020|train|11619900|   0.8992850854219835|
|2021|train| 7108266

## Export to CSV

Parquet stays the canonical output — typed, compressed, partition-pruned. These
copies exist for tools that read a plain file (pandas, scikit-learn, Excel).

`coalesce(1)` forces one part-file per split so each lands as a single readable
`.csv` rather than a directory of fragments. That routes all rows through one
task, so it is the slowest step here.


In [5]:
# export train/test to a single CSV each
import shutil
from pathlib import Path

# year and month are partition columns in the Parquet layout, so they must be
# selected explicitly to survive into a flat file
CSV_COLUMNS = [
    "station_id",
    "start_date",
    "hour",
    "trip_count",
    "year",
    "month",
    "quarter",
    "day_of_week",
    "is_weekend",
    "is_holiday",
    "hour_sin",
    "hour_cos",
    "dow_sin",
    "dow_cos",
    "lag_1h",
    "lag_24h",
    "lag_168h",
    "roll_mean_24h",
    "roll_mean_168h",
    "member_ratio",
]


def export_csv(df, name: str) -> None:
    """Write df as exactly one .csv at {FEATURED}/{name}.csv"""
    staging = f"{FEATURED}/_csv_{name}"
    (
        df.select(*CSV_COLUMNS)
        .orderBy("station_id", "start_date", "hour")
        .coalesce(1)
        .write.mode("overwrite")
        .option("header", True)
        # member_ratio is null on zero-trip hours; write it as an empty field
        # rather than the string "null" so pandas reads it as NaN
        .option("nullValue", "")
        .csv(staging)
    )

    # the write produces a directory; lift the single part-file out of it
    part = next(Path(staging).glob("part-*.csv"))
    target = Path(f"{FEATURED}/{name}.csv")
    shutil.move(str(part), target)
    shutil.rmtree(staging)
    print(f"{name}.csv : {target.stat().st_size / 1e6:,.1f} MB")


export_csv(train, "train")
export_csv(test, "test")

# confirm the header and first rows are readable
print()
for name in ("train", "test"):
    with open(f"{FEATURED}/{name}.csv") as fh:
        print(f"--- {name}.csv")
        for _ in range(3):
            print("   ", fh.readline().rstrip())


train.csv : 7,240.8 MB
test.csv : 1,858.4 MB

--- train.csv
    station_id,start_date,hour,trip_count,year,month,quarter,day_of_week,is_weekend,is_holiday,hour_sin,hour_cos,dow_sin,dow_cos,lag_1h,lag_24h,lag_168h,roll_mean_24h,roll_mean_168h,member_ratio
    7000,2019-01-04,12,0,2019,1,1,5,0,0,1.2246467991473532E-16,-1.0,-0.433883739117558,-0.9009688679024191,12,0,0,2.1666666666666665,1.380952380952381,
    7000,2019-01-04,12,0,2019,1,1,5,0,0,1.2246467991473532E-16,-1.0,-0.433883739117558,-0.9009688679024191,0,0,0,2.1666666666666665,1.380952380952381,
--- test.csv
    station_id,start_date,hour,trip_count,year,month,quarter,day_of_week,is_weekend,is_holiday,hour_sin,hour_cos,dow_sin,dow_cos,lag_1h,lag_24h,lag_168h,roll_mean_24h,roll_mean_168h,member_ratio
    7000,2023-01-01,0,2,2023,1,1,7,1,0,0.0,1.0,-0.7818314824680299,0.6234898018587334,1,0,0,0.5416666666666666,0.6488095238095238,0.0
    7000,2023-01-01,0,2,2023,1,1,7,1,0,0.0,1.0,-0.7818314824680299,0.6234898018587334,2,0,0,0.541666